# Free GPU benchmark: vLLM + Docs Assistant on an NVIDIA T4

Runs on **Kaggle Notebooks** (free, ~30 GPU hours/week) or **Google Colab** free tier. Costs nothing.

**Kaggle setup:** Settings → Accelerator → **GPU T4 x2** (one GPU is used), Settings → Internet → **On**
(phone verification needed once). **Colab:** Runtime → Change runtime type → **T4 GPU**.

What this notebook measures (copy the numbers into your README and resume):
1. vLLM throughput and latency at increasing concurrency → shows continuous batching
2. Retrieval accuracy (hit@1 / hit@4) on the docs question set
3. End-to-end latency of the Docs Assistant, simple RAG vs agent with evidence check

Runtime: about 30–45 minutes, most of it installing vLLM and embedding the docs.

In [ ]:
# 0. Confirm you got a GPU (should list a Tesla T4)
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# 1. Get the project. Change this to YOUR GitHub repo once you've pushed it.
REPO_URL = "https://github.com/Rushikareddy23/docs-rag-inference.git"
import os
if not os.path.exists("docs-rag-inference"):
    !git clone -q {REPO_URL}
%cd docs-rag-inference
!ls

In [ ]:
# 2. Install vLLM (5-10 min) and the services' dependencies
!pip install -q vllm
!pip install -q -r services/retriever/requirements.txt -r services/gateway/requirements.txt
import vllm; print("vLLM", vllm.__version__)

In [ ]:
# 3. Start the vLLM OpenAI-compatible server in the background.
# T4 has no bfloat16, so we use float16 ("half").
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
import subprocess, time, requests
vllm_log = open("vllm.log", "w")
vllm_proc = subprocess.Popen(
    ["vllm", "serve", MODEL, "--dtype", "half", "--max-model-len", "4096",
     "--gpu-memory-utilization", "0.85", "--port", "8000"],
    stdout=vllm_log, stderr=subprocess.STDOUT)
for i in range(120):
    try:
        if requests.get("http://localhost:8000/health", timeout=2).ok:
            print(f"vLLM ready after {i*5}s"); break
    except requests.RequestException:
        pass
    time.sleep(5)
else:
    print(open("vllm.log").read()[-3000:])

In [ ]:
# 4. Benchmark vLLM directly: throughput and latency vs. concurrency
!python bench/bench_llm.py --base-url http://localhost:8000/v1 --model {MODEL} \
    --concurrency 1 4 8 16 32 --requests 64 --max-tokens 128 \
    --label "Tesla T4 (Kaggle free tier)" --out results/llm_t4.json

In [ ]:
# 5. Fetch the docs, build the embedding index, start the retriever (~10 min on Kaggle CPUs)
!bash scripts/fetch_docs.sh
import os, subprocess, time, requests
env = dict(os.environ, DOCS_DIR="data/docs", INDEX_DIR="data/index", EMBED_BATCH="16")
ret_proc = subprocess.Popen(["python", "-m", "uvicorn", "retriever.main:app", "--port", "8001"],
                            cwd="services/retriever", env=env,
                            stdout=open("retriever.log", "w"), stderr=subprocess.STDOUT)
for i in range(240):
    try:
        r = requests.get("http://localhost:8001/healthz", timeout=2)
        if r.ok: print(r.json()); break
    except requests.RequestException:
        pass
    time.sleep(5)

In [ ]:
# 6. Retrieval accuracy: is the right doc in the top 1 / top 4? Compares dense, keyword (BM25) and hybrid search
!python bench/eval_retrieval.py --url http://localhost:8001 --k 1 4 --mode dense bm25 hybrid --out results/retrieval.json

In [ ]:
# 7. Start the gateway pointed at vLLM and the retriever, then ask a real question
env = dict(os.environ, LLM_BASE_URL="http://localhost:8000/v1", LLM_MODEL=MODEL,
           RETRIEVER_URL="http://localhost:8001")
gw_proc = subprocess.Popen(["python", "-m", "uvicorn", "gateway.main:app", "--port", "8080"],
                           cwd="services/gateway", env=env,
                           stdout=open("gateway.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
import json
r = requests.post("http://localhost:8080/v1/ask", timeout=300,
                  json={"question": "How does Triton's dynamic batcher combine requests?", "mode": "agent"})
print(json.dumps(r.json(), indent=2)[:3000])

In [ ]:
# 8. End-to-end benchmark: simple RAG vs. agent with evidence check
%cd bench
!python bench_gateway.py --url http://localhost:8080 --n 20 --out ../results/gateway_t4.json
%cd ..

In [ ]:
# 9. Summary to copy into your README / resume, and a zip of all results to download
import json
llm = json.load(open("results/llm_t4.json"))
rows = llm["results"]
best = max(rows, key=lambda r: r["output_tokens_per_s"])
ret_all = json.load(open("results/retrieval.json"))
ret = ret_all["hybrid"]["summary"]
gw = json.load(open("results/gateway_t4.json"))
print(f"Model: {llm['model']} on {llm['label']}")
print(f"Peak throughput: {best['output_tokens_per_s']} tok/s at concurrency {best['concurrency']} "
      f"({best['output_tokens_per_s'] / rows[0]['output_tokens_per_s']:.1f}x vs concurrency 1)")
print(f"TTFT p95 at concurrency {best['concurrency']}: {best['ttft_p95_ms']} ms")
for m, r in ret_all.items():
    print(f"Retrieval ({m}): hit@1 {r['summary']['hit@1']:.0%}, hit@4 {r['summary']['hit@4']:.0%} over {r['summary']['questions']} questions")
for mode, g in gw.items():
    print(f"{mode}: p50 {g['latency_p50_ms']} ms, p95 {g['latency_p95_ms']} ms, {g['avg_llm_calls']} LLM calls/answer")
!zip -qr results.zip results && echo "Download results.zip from the file browser (Kaggle: /kaggle/working/docs-rag-inference)"